### Homework: Language Models, Attention, and Transformer Training

#### Instructions:
- Complete the following exercises based on the lecture material.
- Submit your completed notebook with all cells executed.

---

## Part 1: Language Models and Attention

### Question 1: Understanding Attention Mechanisms
**(a)** Explain the difference between hard and soft attention mechanisms in NLP.

Soft attention computes a weight for every input token, then takes a weighted average allowing it to use all items and focus on only some of then. Hard attention selects a single item instead of averaging all of them. Soft attention is more widely used and is easier during training because it is differentiable, while hard attention may be more effecient and useful for certain workloads but requires work arounds for back propagatation during training.

**(b)** Given an input sequence of three words, describe how the attention scores are computed using scaled dot-product attention.

Each input token, which is a word here, is linearly transformed into a key, query, and value. For each query, the dot product is computed with every key and normalized by dividing by the square root of the query depth. The scores produced by this operation are converted to weights using softmax, and finally the value is matrix multiplied by the attention weights to produce the output token. Because there are three words in the input, a 3x3 matrix of scores is produced with each query scored against each key


### Question 2: Implementing Scaled Dot-Product Attention
Modify the given function to compute attention weights for a given query, key, and value matrix.


In [13]:
import torch
import torch.nn.functional as F

def scaled_dot_product_attention(query, key, value):
    d_k = query.shape[-1]
    scores = torch.matmul(query, key.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k))

    # upper triangle excluding the diagonal
    seq_len_q = query.shape[-2]
    seq_len_k = key.shape[-2]
    causal_mask = torch.triu(
        torch.ones(seq_len_q, seq_len_k, dtype=torch.bool),
        diagonal=1
    )
    # apply mask by adding very negative values where blocked
    scores = scores.masked_fill(causal_mask, torch.finfo(scores.dtype).min)

    attention_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attention_weights, value)

query = torch.rand(1, 3, 64)
key = torch.rand(1, 3, 64)
value = torch.rand(1, 3, 64)

output = scaled_dot_product_attention(query, key, value)
print(output.shape)

tensor([[[1.0000, 0.0000, 0.0000],
         [0.4865, 0.5135, 0.0000],
         [0.2931, 0.3144, 0.3925]]])
torch.Size([1, 3, 64])


Modify the function to include masking for causal attention.

---

## Part 2: The Transformer Architecture

### Question 3: Understanding Self-Attention
**(a)** Explain the role of self-attention in Transformer models. How does it differ from traditional sequence models like RNNs?

The self-attention mechanism allows transformer models to focus on the most relevant parts of the input sequence for each token by producing contextualized token representations. This differs from traditonal sequence models like RNNs because the input does not need to be processed sequentially, instead the attention for each input token can be computed in a non-linear manner. Self-attention also enables longer term dependancies to be modeled because the entire input sequence is considered for each input token.

**(b)** Why is positional encoding necessary in Transformers? Implement a simple sinusoidal positional encoding function.

Positional encoding is necessary in Transformers because the input tokens are not processed in order. This means that the model needs a way to understand the relative and absolute positions of tokens within the sequence. Different positional encoding functions are used depending on the requirements of the workload.

In [10]:
import numpy as np

def positional_encoding(seq_len, d_model):
    position = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pe = np.zeros((seq_len, d_model))
    pe[:, 0::2] = np.sin(position * div_term)
    pe[:, 1::2] = np.cos(position * div_term)
    return pe

pos_enc = positional_encoding(10, 64)
print(pos_enc.shape)

(10, 64)


---

## Part 3: Autoregressive Training for Transformer LLMs

### Question 4: Implementing a Transformer Decoder Block
Write a function that implements a Transformer decoder block with self-attention and a feed-forward network.

In [9]:
import torch.nn as nn

class TransformerDecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(embed_dim=d_model, num_heads=num_heads)
        self.feed_forward = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

        # drops attention output
        self.dropout_attn = nn.Dropout(dropout)
        # drops feed forward output
        self.dropout_ffn = nn.Dropout(dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        attn_output, _ = self.self_attn(x, x, x)
        x = self.norm1(x + self.dropout_attn(attn_output))
        ff_output = self.feed_forward(x)
        return self.norm2(x + self.dropout_ffn(ff_output))

# Test with dummy input
decoder = TransformerDecoderBlock(64, 8, 256)
x = torch.rand(10, 32, 64)  # (seq_len, batch_size, d_model)
out = decoder(x)
print(out.shape)

torch.Size([10, 32, 64])


Modify the function to include dropout layers.

---

### Submission
Submit your completed notebook with answers and executed code outputs. Ensure that all plots, tokenized outputs, and model results are included in your submission.